In [64]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [65]:
!pip install dagshub

In [66]:
!pip install mlflow

In [71]:
import numpy as np
import pandas as pd
import mlflow
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("DAGSHUB_USERNAME")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/ml_assignment_1.mlflow")

train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

test_ids = test['Id']

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [72]:
# Fill None features — same as train
none_features = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

for col in none_features:
    if col in train.columns:
        train[col] = train[col].fillna('None')
    if col in test.columns:
        test[col] = test[col].fillna('None')

# Fill categorical with mode
train['MasVnrType'] = train['MasVnrType'].fillna(train['MasVnrType'].mode()[0])
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
test['MasVnrType'] = test['MasVnrType'].fillna(test['MasVnrType'].mode()[0])

# Fill numerical
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())

test['MasVnrArea'] = test['MasVnrArea'].fillna(0)
test['GarageYrBlt'] = test['GarageYrBlt'].fillna(0)
test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
test['LotFrontage'] = test['LotFrontage'].fillna(test['LotFrontage'].median())

# Fill remaining test nulls with mode
cat_cols_test = test.select_dtypes(include='object').columns
for col in cat_cols_test:
    test[col] = test[col].fillna(test[col].mode()[0])

num_cols_test = test.select_dtypes(include=[np.number]).columns
test[num_cols_test] = test[num_cols_test].fillna(test[num_cols_test].median())

print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())

Train missing: 0
Test missing: 0


In [73]:
# Area features
for df in [train, test]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBathrooms'] = (df['FullBath'] + 0.5 * df['HalfBath'] +
                            df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['YearsSinceRemodel'] = df['YearsSinceRemodel'].clip(lower=0)
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['OverallScore'] = df['OverallQual'] * df['OverallCond']
    df['QualXSF'] = df['OverallQual'] * df['TotalSF']

print("✅ Feature engineering done")

✅ Feature engineering done


In [74]:
# Ordinal encoding
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond', 'PoolQC']

for col in ordinal_cols:
    train[col] = train[col].map(quality_map).fillna(0).astype(int)
    test[col] = test[col].map(quality_map).fillna(0).astype(int)

# Combine train and test for consistent one-hot encoding
target = train['SalePrice']
train = train.drop(columns=['SalePrice'])

combined = pd.concat([train, test], axis=0).reset_index(drop=True)
cat_cols = combined.select_dtypes(include='object').columns.tolist()
combined = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

# Split back
train = combined.iloc[:len(train)].copy()
test = combined.iloc[len(train):].copy()

train['SalePrice'] = target.values

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 242)
Test shape: (1459, 241)


In [75]:
# Feature selection — same as experiment notebook
X = train.drop(columns=['SalePrice', 'Id'])
y = train['SalePrice']

correlations = X.corrwith(y).abs().sort_values(ascending=False)
weak_features = correlations[correlations < 0.05].index.tolist()

X = X.drop(columns=weak_features)
test_X = test.drop(columns=['Id'], errors='ignore')
test_X = test_X.drop(columns=weak_features, errors='ignore')

# Drop duplicate/correlated columns
cols_to_drop = [
    'BsmtFinType1_None',
    'TotalBath',
    'GarageType_None',
    'GarageFinish_None',
    'YearBuilt',
    'YearRemodAdd',
    'GarageYrBlt',
]

X = X.drop(columns=cols_to_drop, errors='ignore')
test_X = test_X.drop(columns=cols_to_drop, errors='ignore')

print("X shape:", X.shape)
print("Test shape:", test_X.shape)

X shape: (1460, 147)
Test shape: (1459, 147)


In [79]:
from sklearn.preprocessing import StandardScaler

# Scale test_X using train statistics
scaler = StandardScaler()
scaler.fit(X)  # fit on full train X
test_X_scaled = scaler.transform(test_X)

# Load model version 4 (no scaler inside)
model = mlflow.sklearn.load_model("models:/house_prices_best_model/4")

# Predict
predictions_log = model.predict(test_X_scaled)
predictions = np.expm1(predictions_log)

print("Sample predictions:", predictions[:5])
print("Min price:", predictions.min())
print("Max price:", predictions.max())

Sample predictions: [132962.80582797 165451.91224984 166739.86327888 186094.5158593
 193785.57731124]
Min price: 47954.27664427362
Max price: 1292013.7404140248


In [80]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('/kaggle/working/submission3.csv', index=False)
print("✅ Submission saved")
print(submission.head(10))

✅ Submission saved
     Id      SalePrice
0  1461  132962.805828
1  1462  165451.912250
2  1463  166739.863279
3  1464  186094.515859
4  1465  193785.577311
5  1466  161105.552683
6  1467  163873.358096
7  1468  153867.018098
8  1469  185204.215334
9  1470  124432.575743


In [81]:
pipeline_v3 = mlflow.sklearn.load_model("models:/house_prices_best_model/3")

# Check what columns pipeline expects
pipeline_scaler = pipeline_v3.named_steps['scaler']
print("Pipeline expects:", pipeline_scaler.n_features_in_, "features")
print("Our test_X has:", test_X.shape[1], "features")

Pipeline expects: 147 features
Our test_X has: 147 features


In [83]:
# Get feature names from pipeline scaler
feature_names = pipeline_v3.named_steps['scaler'].feature_names_in_

# Reorder test_X to match exact order
test_X_aligned = test_X.reindex(columns=feature_names, fill_value=0)

predictions_log = pipeline_v3.predict(test_X_aligned)
predictions = np.expm1(predictions_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('/kaggle/working/submission4.csv', index=False)
print("✅ Submission saved")
print(submission.head(5))

✅ Submission saved
     Id      SalePrice
0  1461  117179.843760
1  1462  151299.923708
2  1463  177033.387957
3  1464  197781.630773
4  1465  196336.664891
